# Comparison study using Persistent homology with distance filtration -Evaluation

For comparison, we implement a similar pipeline using distance filtration as opposed to signed distance filtration to analyze the data. For ease of computation, computing the distance persistence diagram is equivalent to taking the absolute values of the birth and death times of points in the persistence diagram.

In [71]:
# load packages and data
import numpy as np
import pickle
import math
import time
import sys
module_paths = ['']
for path in module_paths :
    if path not in sys.path:
        sys.path.append(path)
from utils_load_PHloc import datasets_of_interest, injected_datasets_of_interest, levels_of_interest
import matplotlib.pyplot as plt
import pomegranate
import sklearn
import torch
from pomegranate.gmm import GeneralMixtureModel
from pomegranate.distributions import *
from joblib import Parallel, delayed
from sklearn.model_selection import KFold
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import pandas as pd 
from sklearn.utils import resample

PH_folder = '' 
anatomy = 'knee'
filepath = PH_folder + 'PH_all_{}.pkl'.format(anatomy)
PH_all_datasets = pickle.load(open(filepath, 'rb'))

# threshold value
THR = .5
# generate a new dictionary for truncated datasets
truncated_PH_all_datasets = {}
for i in datasets_of_interest:
    diagram = PH_all_datasets[i]
    truncated_PH_all_datasets[i] = diagram[diagram[:,1] >= diagram[:,0] + THR]
    
names = [injected_datasets_of_interest[i]+"_"+[str(x) for x in datasets_of_interest][i] for i in range(27)]

labels = ['CTRL_0%(1)', 'CTRL_0%(2)', 'CTRL_0%(3)', 'CTRL_0%(4)', 
          'U937_1%(1)', 'U937_1%(2)', 'U937_7%', 'U937_8%', 'U937_10%(1)', 'U937_10%(2)', 'U937_10%(3)', 
          'HL60_23%', 'HL60_25%(1)', 'HL60_25%(2)', 
          'P1_10%', 'P1_40%', 'P1_44%', 'P1_51%', 'P1_60%', 'P1_76%', 
          'P2_59%', 'P2_88%', 'P2_90%', 
          'MNC_53%', 'MNC_67%', 'MNC_75%', 'MNC_86%']

phases_of_interest = [0,0,0,0,
         1,1,1,1,1,1,1,
         1,1,1,
         1,2,2,2,2,1,
         2,2,2,
         1,2,2,2]

name_phase = [labels[i]+" [Phase "+str(phases_of_interest[i])+"]" for i in range(27)]

# Binning parameters
XLIMS = np.array([[0,15],[0,10],[0,20]])
YLIMS = np.array([[0,8],[0,15],[0,20]])
NB_BINS_PER_SIDE = 100

pds = [np.abs(truncated_PH_all_datasets[i]) for i in datasets_of_interest]

In [74]:
# function that extract quadrant data according to the name
def extract_dim(dim_str):
    pds_dim = []
    dim = int(dim_str[2])
    for i in range(27):
        current_pd = pds[i][pds[i][:,1]-pds[i][:,0]>0,:]
        pds_dim.append(current_pd[current_pd[:,2]==dim,:2])
    return pds_dim

# dictionary for number of components for each phase model
num_component_dict = {"ph0":[3,4,5], "ph2":[4,2,2], "ph1":[3,4,5]}

In [76]:
def kl_divergence(model1, model2, dim):
    x = np.linspace(XLIMS[dim][0],XLIMS[dim][1],100)
    y = np.linspace(YLIMS[dim][0],YLIMS[dim][1],100)
    xx,yy = np.meshgrid(x,y)
    x_ = np.array(list(zip(xx.flatten(), yy.flatten())))

    p1 = model1.probability(x_).reshape(len(x),len(y))
    p2 = model2.probability(x_).reshape(len(x),len(y))
     
    p1 = p1/np.sum(p1)
    p2 = p2/np.sum(p2)
    
    return np.sum(p1*(np.log(p1/p2))) 

# Hellinger distance
def Hellinger(model1, model2, dim):
    x = np.linspace(XLIMS[dim][0],XLIMS[dim][1],100)
    y = np.linspace(YLIMS[dim][0],YLIMS[dim][1],100)
    xx,yy = np.meshgrid(x,y)
    x_ = np.array(list(zip(xx.flatten(), yy.flatten())))

    p1 = model1.probability(x_).reshape(len(x),len(y))
    p2 = model2.probability(x_).reshape(len(x),len(y))
     
    p1 = p1/np.sum(p1)
    p2 = p2/np.sum(p2)
    
    summation = np.sum(np.square(np.sqrt(p1)-np.sqrt(p2)))   
    
    return np.sqrt(summation)/np.sqrt(2)

In [90]:
# --- Parallelized LOOCV Function ---
def process_fold_simple(train_index, test_index, X, y, n_components, phases, dim, num_subjects):
    X_train_list = [X[ind] for ind in train_index] # Keep as list of arrays
    X_test_single = np.array(X[test_index[0]]) # Test data for one subject
    y_train = y[train_index]
    y_test = y[test_index][0] # y_test is a single value

    # Prepare data for each phase model
    phase_data = {}
    for p in phases:
        # Filter X_train_list based on y_train
        current_phase_samples = [X_train_list[i] for i in range(len(X_train_list)) if y_train[i] == p]
        #print(f"length for phase {p}: {len(current_phase_samples)}")
        if current_phase_samples:
            phase_data[p] = np.vstack(current_phase_samples)
        else:
            phase_data[p] = np.array([]) 
            print("empty")

    # Fit GMMs for each phase
    models = {}
    for p_idx, p in enumerate(phases):
        if phase_data[p].shape[0] > 0:
            try:
                model = GeneralMixtureModel.from_samples(
                    pomegranate.MultivariateGaussianDistribution,
                    n_components=n_components[p_idx],
                    X=phase_data[p]
                )
                model.fit(
                    X=phase_data[p],
                    weights=np.abs(phase_data[p][:,1] - phase_data[p][:,0]), 
                    stop_threshold=.001,
                    verbose=False # Suppress verbose output during fitting
                )
                models[p] = model
            except Exception as e:
                # Handle cases where GMM fitting might fail (e.g., too few samples)
                # print(f"Warning: GMM fitting failed for phase {p} in a fold: {e}")
                models[p] = None
        else:
            models[p] = None
            
    print(models)

    # --- Prepare test models for Hellinger and KL-Divergence ---
    test_models = {}
    for p_idx, p in enumerate(phases):
        if X_test_single.shape[0] > 0:
            try:
                test_model = GeneralMixtureModel.from_samples(
                    pomegranate.MultivariateGaussianDistribution,
                    n_components=n_components[p_idx],
                    X=X_test_single
                )
                test_model.fit(
                    X=X_test_single,
                    weights=np.abs(X_test_single[:,1]-X_test_single[:,0]), 
                    stop_threshold=.001,
                    verbose=True
                )
                test_models[p] = test_model
            except Exception as e:
                print(f"Warning: Test GMM fitting failed for phase {p} in a fold: {e}")
                test_models[p] = None
        else:
            test_models[p] = None

    # --- Prediction via Hellinger distance ---
    hdist = []
    for p in phases:
        if models[p] and test_models[p]:
            hdist.append(Hellinger(models[p], test_models[p], dim))
        else:
            hdist.append(np.inf) # Use inf if a model is missing or could not be fitted

    y_pred_h = phases[np.argmin(hdist)] if not np.all(np.isinf(hdist)) else -1

    # --- Prediction via KL-divergence ---
    kldiv = []
    for p in phases:
        if models[p] and test_models[p]:
            kldiv.append(kl_divergence(models[p], test_models[p], dim))
        else:
            kldiv.append(np.inf) # Use inf if a model is missing or could not be fitted

    y_pred_k = phases[np.argmin(kldiv)] if not np.all(np.isinf(kldiv)) else -1

    return y_test, y_pred_h, y_pred_k

In [94]:
# --- Main execution loop for multiple regions ---
regions_to_analyze = ["ph0", "ph1", "ph2"]
all_results = {}

for region in regions_to_analyze:
    print(f"\n--- Processing Region: {region} ---")

    X = extract_dim(region)
    y = np.array([0,0,0,0,
         1,1,1,1,1,1,1,
         1,1,1,
         1,2,2,2,2,1,
         2,2,2,
         1,2,2,2])
    n_components = num_component_dict[region]
    num_subjects = 27 
    loocv = LeaveOneOut()
    phases = [0, 1, 2] # class labels
    dim = int(region[2]) 

    # Parallel execution for the current region
    fold_results = Parallel(n_jobs=-1)(
        delayed(process_fold_simple)(train_index, test_index, X, y, n_components, phases, dim, num_subjects)
        for i, (train_index, test_index) in enumerate(loocv.split(X))
    )

    # Unpack results for the current region
    y_test_true_region = [res[0] for res in fold_results]
    hellinger_pred_region = [res[1] for res in fold_results]
    kl_divergence_pred_region = [res[2] for res in fold_results]
        
    filtered_y_test_h = [y_t for y_t, y_p in zip(y_test_true_region, hellinger_pred_region) if y_p != -1]
    filtered_y_pred_h = [y_p for y_p in hellinger_pred_region if y_p != -1]
    if len(filtered_y_pred_h)!=27:
        print("missing prediction from hellinger")
        
    filtered_y_test_kl = [y_t for y_t, y_p in zip(y_test_true_region, kl_divergence_pred_region) if y_p != -1]
    filtered_y_pred_kl = [y_p for y_p in kl_divergence_pred_region if y_p != -1]
    if len(filtered_y_pred_kl)!=27:
        print("missing prediction from kl")
        

    # --- Calculate and store metrics for the current region ---
    region_metrics = {}

    # Hellinger Distance Metrics
    if filtered_y_pred_h:
        region_metrics['hellinger'] = {
            'accuracy': accuracy_score(y_test_true_region, hellinger_pred_region),
            'f1_macro': f1_score(y_test_true_region, hellinger_pred_region, average='macro', zero_division=0),
            'confusion_matrix': confusion_matrix(y_test_true_region, hellinger_pred_region, labels=phases),
            'classification_report': classification_report(y_test_true_region, hellinger_pred_region, labels=phases, target_names=[f'Phase {p}' for p in phases], zero_division=0, output_dict=True)
        }
    else:
        region_metrics['hellinger'] = {'accuracy': 0, 'f1_macro': 0, 'confusion_matrix': None, 'classification_report': None}

    # KL-Divergence Metrics
    if filtered_y_pred_kl:
        region_metrics['kl_divergence'] = {
            'accuracy': accuracy_score(y_test_true_region, kl_divergence_pred_region),
            'f1_macro': f1_score(y_test_true_region, kl_divergence_pred_region, average='macro', zero_division=0),
            'confusion_matrix': confusion_matrix(y_test_true_region, kl_divergence_pred_region, labels=phases),
            'classification_report': classification_report(y_test_true_region, kl_divergence_pred_region, labels=phases, target_names=[f'Phase {p}' for p in phases], zero_division=0, output_dict=True)
        }
    else:
        region_metrics['kl_divergence'] = {'accuracy': 0, 'f1_macro': 0, 'confusion_matrix': None, 'classification_report': None}

    all_results[region] = region_metrics


# --- Display Results for All Regions ---
print("\n" + "="*50)
print("             OVERALL RESULTS SUMMARY             ")
print("="*50)

for region, metrics in all_results.items():
    print(f"\n### Region: {region} ###")

    for method, data in metrics.items():
        print(f"\n--- Method: {method.replace('_', ' ').title()} ---")
        print(f"Accuracy: {data['accuracy']:.4f}")
        print(f"F1-Macro Score: {data['f1_macro']:.4f}")

        if data['confusion_matrix'] is not None:
            print("\nConfusion Matrix:")
            cm_df = pd.DataFrame(data['confusion_matrix'],
                                 index=[f'True Phase {p}' for p in phases],
                                 columns=[f'Pred Phase {p}' for p in phases])
            print(cm_df)
        else:
            print("\nConfusion Matrix: Not available (no valid predictions)")

        if data['classification_report'] is not None:
            print("\nClassification Report:")
            # Convert classification report dict to a nice DataFrame for display
            report_df = pd.DataFrame(data['classification_report']).transpose()
            print(report_df)
        else:
            print("\nClassification Report: Not available (no valid predictions)")

    print("\n" + "-"*40)


--- Processing Region: ph0 ---

--- Processing Region: ph1 ---

--- Processing Region: ph2 ---

             OVERALL RESULTS SUMMARY             

### Region: ph0 ###

--- Method: Hellinger ---
Accuracy: 0.6296
F1-Macro Score: 0.6600

Confusion Matrix:
              Pred Phase 0  Pred Phase 1  Pred Phase 2
True Phase 0             4             0             0
True Phase 1             1             7             5
True Phase 2             1             3             6

Classification Report:
              precision    recall  f1-score   support
Phase 0        0.666667  1.000000  0.800000   4.00000
Phase 1        0.700000  0.538462  0.608696  13.00000
Phase 2        0.545455  0.600000  0.571429  10.00000
accuracy       0.629630  0.629630  0.629630   0.62963
macro avg      0.637374  0.712821  0.660041  27.00000
weighted avg   0.637823  0.629630  0.623234  27.00000

--- Method: Kl Divergence ---
Accuracy: 0.5926
F1-Macro Score: 0.6278

Confusion Matrix:
              Pred Phase 0  Pred P

# LOOCV with Bootstrap

In [95]:
# --- Parallelized LOOCV Function ---
def process_fold_full_bootstrap(train_index, test_index, X, y, n_components, phases, dim, num_subjects):
    X_train_list = [X[ind] for ind in train_index] # Keep as list of arrays
    X_test_single = np.array(X[test_index[0]]) # Test data for one subject
    y_train = y[train_index]
    y_test = y[test_index][0] # y_test is a single value

    # Prepare data for each phase model
    phase_data = {}
    for p in phases:
        # Filter X_train_list based on y_train
        current_phase_samples = [X_train_list[i] for i in range(len(X_train_list)) if y_train[i] == p]
        if current_phase_samples:
            data = np.vstack(current_phase_samples)
            n_sub = data.shape[0]
            samples = []
            for iters in range(50):
                samples.append(resample(data, n_samples=int(np.floor(n_sub/10)), replace=True))
            phase_data[p] = np.vstack(samples)
        else:
            phase_data[p] = np.array([]) # Empty array if no data for this phase

    # Fit GMMs for each phase
    models = {}
    for p_idx, p in enumerate(phases):
        if phase_data[p].shape[0] > 0:
            try:
                model = GeneralMixtureModel.from_samples(
                    pomegranate.MultivariateGaussianDistribution,
                    n_components=n_components[p_idx],
                    X=phase_data[p]
                )
                model.fit(
                    X=phase_data[p],
                    weights=np.abs(phase_data[p][:,1] - phase_data[p][:,0]),
                    stop_threshold=.001,
                    verbose=False # Suppress verbose output during fitting
                )
                models[p] = model
            except Exception as e:
                # Handle cases where GMM fitting might fail (e.g., too few samples)
                # print(f"Warning: GMM fitting failed for phase {p} in a fold: {e}")
                models[p] = None
        else:
            models[p] = None

    
    # --- Prepare test models for Hellinger and KL-Divergence ---
    test_models = {}
    for p_idx, p in enumerate(phases):
        if X_test_single.shape[0] > 0:
            try:
                test_model = GeneralMixtureModel.from_samples(
                    pomegranate.MultivariateGaussianDistribution,
                    n_components=n_components[p_idx],
                    X=X_test_single
                )
                test_model.fit(
                    X=X_test_single,
                    weights=np.abs(X_test_single[:,1] - X_test_single[:,0]), 
                    stop_threshold=.001,
                    verbose=False
                )
                test_models[p] = test_model
            except Exception as e:
                # print(f"Warning: Test GMM fitting failed for phase {p} in a fold: {e}")
                test_models[p] = None
        else:
            test_models[p] = None

    # --- Prediction via Hellinger distance ---
    hdist = []
    for p in phases:
        if models[p] and test_models[p]:
            hdist.append(Hellinger(models[p], test_models[p], dim))
        else:
            hdist.append(np.inf) # Use inf if a model is missing or could not be fitted

    y_pred_h = phases[np.argmin(hdist)] if not np.all(np.isinf(hdist)) else -1

    # --- Prediction via KL-divergence ---
    kldiv = []
    for p in phases:
        if models[p] and test_models[p]:
            kldiv.append(kl_divergence(models[p], test_models[p], dim))
        else:
            kldiv.append(np.inf) # Use inf if a model is missing or could not be fitted

    y_pred_k = phases[np.argmin(kldiv)] if not np.all(np.isinf(kldiv)) else -1

    return y_test, y_pred_h, y_pred_k

In [97]:
# --- Main execution loop for multiple regions ---
regions_to_analyze = ["ph0", "ph1", "ph2"]
all_results_2 = {}

for region in regions_to_analyze:
    print(f"\n--- Processing Region: {region} ---")

    X = extract_dim(region)
    y = np.array([0,0,0,0,
         1,1,1,1,1,1,1,
         1,1,1,
         1,2,2,2,2,1,
         2,2,2,
         1,2,2,2])
    n_components = num_component_dict[region]
    num_subjects = 27 
    loocv = LeaveOneOut()
    phases = [0, 1, 2] # class labels
    dim = int(region[2]) 

    # Parallel execution for the current region
    fold_results = Parallel(n_jobs=-1)(
        delayed(process_fold_full_bootstrap)(train_index, test_index, X, y, n_components, phases, dim, num_subjects)
        for i, (train_index, test_index) in enumerate(loocv.split(X))
    )

    # Unpack results for the current region
    y_test_true_region = [res[0] for res in fold_results]
    hellinger_pred_region = [res[1] for res in fold_results]
    kl_divergence_pred_region = [res[2] for res in fold_results]

    # Filter out unclassifiable predictions for metric calculation (predictions of -1)
    # This ensures metrics are only calculated on cases where a prediction was actually made        
    filtered_y_test_h = [y_t for y_t, y_p in zip(y_test_true_region, hellinger_pred_region) if y_p != -1]
    filtered_y_pred_h = [y_p for y_p in hellinger_pred_region if y_p != -1]
    if len(filtered_y_pred_h)!=27:
        print("missing prediction from hellinger")
        
    filtered_y_test_kl = [y_t for y_t, y_p in zip(y_test_true_region, kl_divergence_pred_region) if y_p != -1]
    filtered_y_pred_kl = [y_p for y_p in kl_divergence_pred_region if y_p != -1]
    if len(filtered_y_pred_kl)!=27:
        print("missing prediction from kl")
        

    # --- Calculate and store metrics for the current region ---
    region_metrics = {}

    # Hellinger Distance Metrics
    if filtered_y_pred_h:
        region_metrics['hellinger'] = {
            'accuracy': accuracy_score(filtered_y_test_h, filtered_y_pred_h),
            'f1_macro': f1_score(filtered_y_test_h, filtered_y_pred_h, average='macro', zero_division=0),
            'confusion_matrix': confusion_matrix(filtered_y_test_h, filtered_y_pred_h, labels=phases),
            'classification_report': classification_report(filtered_y_test_h, filtered_y_pred_h, labels=phases, target_names=[f'Phase {p}' for p in phases], zero_division=0, output_dict=True)
        }
    else:
        region_metrics['hellinger'] = {'accuracy': 0, 'f1_macro': 0, 'confusion_matrix': None, 'classification_report': None}

    # KL-Divergence Metrics
    if filtered_y_pred_kl:
        region_metrics['kl_divergence'] = {
            'accuracy': accuracy_score(filtered_y_test_kl, filtered_y_pred_kl),
            'f1_macro': f1_score(filtered_y_test_kl, filtered_y_pred_kl, average='macro', zero_division=0),
            'confusion_matrix': confusion_matrix(filtered_y_test_kl, filtered_y_pred_kl, labels=phases),
            'classification_report': classification_report(filtered_y_test_kl, filtered_y_pred_kl, labels=phases, target_names=[f'Phase {p}' for p in phases], zero_division=0, output_dict=True)
        }
    else:
        region_metrics['kl_divergence'] = {'accuracy': 0, 'f1_macro': 0, 'confusion_matrix': None, 'classification_report': None}

    all_results_2[region] = region_metrics


# --- Display Results for All Regions ---
print("\n" + "="*50)
print("             OVERALL RESULTS SUMMARY             ")
print("="*50)

for region, metrics in all_results_2.items():
    print(f"\n### Region: {region} ###")

    for method, data in metrics.items():
        print(f"\n--- Method: {method.replace('_', ' ').title()} ---")
        print(f"Accuracy: {data['accuracy']:.4f}")
        print(f"F1-Macro Score: {data['f1_macro']:.4f}")

        if data['confusion_matrix'] is not None:
            print("\nConfusion Matrix:")
            cm_df = pd.DataFrame(data['confusion_matrix'],
                                 index=[f'True Phase {p}' for p in phases],
                                 columns=[f'Pred Phase {p}' for p in phases])
            print(cm_df)
        else:
            print("\nConfusion Matrix: Not available (no valid predictions)")

        if data['classification_report'] is not None:
            print("\nClassification Report:")
            # Convert classification report dict to a nice DataFrame for display
            report_df = pd.DataFrame(data['classification_report']).transpose()
            print(report_df)
        else:
            print("\nClassification Report: Not available (no valid predictions)")

    print("\n" + "-"*40)


--- Processing Region: ph0 ---

--- Processing Region: ph1 ---

--- Processing Region: ph2 ---

             OVERALL RESULTS SUMMARY             

### Region: ph0 ###

--- Method: Hellinger ---
Accuracy: 0.6667
F1-Macro Score: 0.7079

Confusion Matrix:
              Pred Phase 0  Pred Phase 1  Pred Phase 2
True Phase 0             3             1             0
True Phase 1             0             9             4
True Phase 2             0             4             6

Classification Report:
              precision    recall  f1-score    support
Phase 0        1.000000  0.750000  0.857143   4.000000
Phase 1        0.642857  0.692308  0.666667  13.000000
Phase 2        0.600000  0.600000  0.600000  10.000000
accuracy       0.666667  0.666667  0.666667   0.666667
macro avg      0.747619  0.680769  0.707937  27.000000
weighted avg   0.679894  0.666667  0.670194  27.000000

--- Method: Kl Divergence ---
Accuracy: 0.6296
F1-Macro Score: 0.6813

Confusion Matrix:
              Pred Phase 0 